In [14]:
import pandas as pd
import os

# Dataset path
file_path = "../data/Bengaluru_House_Data.csv"

# Check whether the dataset exists
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    print("Dataset shape:", df.shape)
    display(df.head())
else:
    print("Dataset not found. Check the CSV filename and location.")

Dataset loaded successfully!
Dataset shape: (13320, 9)


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [15]:
import pandas as pd

df = pd.read_csv("../data/Bengaluru_House_Data.csv")

# Display column names
print("Column Names:")
print(df.columns.tolist())

# Display dataset dimensions
print("\nDataset Shape:")
print(df.shape)

# Display the first five rows
print("\nFirst Five Rows:")
display(df.head())

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

Column Names:
['area_type', 'availability', 'location', 'size', 'society', 'total_sqft', 'bath', 'balcony', 'price']

Dataset Shape:
(13320, 9)

First Five Rows:


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00



Missing Values:
area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64


In [16]:
print("Missing values in each column:\n")
print(df.isnull().sum())

Missing values in each column:

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64


In [17]:
print("Data types:\n")
print(df.dtypes)

Data types:

area_type           str
availability        str
location            str
size                str
society             str
total_sqft          str
bath            float64
balcony         float64
price           float64
dtype: object


In [18]:
# Create a working copy
df_clean = df.copy()

# Remove columns not needed for our first model
df_clean = df_clean.drop(columns=["society", "balcony"])

# Remove rows missing essential values
df_clean = df_clean.dropna(subset=["location", "size", "bath"])

# Remove duplicate records
df_clean = df_clean.drop_duplicates()

# Display results
print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", df_clean.shape)

print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

display(df_clean.head())

Original dataset shape: (13320, 9)
Cleaned dataset shape: (12578, 7)

Missing values after cleaning:


area_type       0
availability    0
location        0
size            0
total_sqft      0
bath            0
price           0
dtype: int64


,area_type,availability,location,size,total_sqft,bath,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056,2.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440,2.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521,3.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200,2.0,51.00


In [19]:
# Extract bedroom count from values such as "2 BHK" and "4 Bedroom"
df_clean["bhk"] = (
    df_clean["size"]
    .str.extract(r"(\d+)")
    .astype(int)
)

# Convert property area to a numeric value
def convert_sqft(value):
    try:
        return float(value)
    except ValueError:
        try:
            low, high = value.split("-")
            return (float(low) + float(high)) / 2
        except ValueError:
            return None

df_clean["total_sqft"] = df_clean["total_sqft"].apply(convert_sqft)

# Remove rows where area conversion failed
df_clean = df_clean.dropna(subset=["total_sqft"])

# Display the results
display(df_clean[["location", "size", "bhk", "total_sqft", "bath", "price"]].head())

print("Dataset shape:", df_clean.shape)
print("\nMissing values:")
print(df_clean.isnull().sum())

,location,size,bhk,total_sqft,bath,price
0,Electronic City Phase II,2 BHK,2,1056.0,2.0,39.07
1,Chikka Tirupathi,4 Bedroom,4,2600.0,5.0,120.00
2,Uttarahalli,3 BHK,3,1440.0,2.0,62.00
3,Lingadheeranahalli,3 BHK,3,1521.0,3.0,95.00
4,Kothanur,2 BHK,2,1200.0,2.0,51.00


Dataset shape: (12532, 8)

Missing values:
area_type       0
availability    0
location        0
size            0
total_sqft      0
bath            0
price           0
bhk             0
dtype: int64


In [20]:
# Select features and target
data = df_clean[["location", "total_sqft", "bath", "bhk", "price"]].copy()

# Remove extra spaces from location names
data["location"] = data["location"].str.strip()

# Remove rows with invalid values
data = data.dropna()

# Display the prepared data
print("Prepared dataset shape:", data.shape)

display(data.head())

print("\nNumber of unique locations:", data["location"].nunique())

Prepared dataset shape: (12532, 5)


,location,total_sqft,bath,bhk,price
0,Electronic City Phase II,1056.0,2.0,2,39.07
1,Chikka Tirupathi,2600.0,5.0,4,120.00
2,Uttarahalli,1440.0,2.0,3,62.00
3,Lingadheeranahalli,1521.0,3.0,3,95.00
4,Kothanur,1200.0,2.0,2,51.00



Number of unique locations: 1287


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Copy prepared data
data = data.copy()

# Group locations with fewer than 10 records
location_counts = data["location"].value_counts()
data["location"] = data["location"].apply(
    lambda loc: loc if location_counts[loc] >= 10 else "Other"
)

# Separate inputs and target
X = data[["location", "total_sqft", "bath", "bhk"]]
y = data["price"]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Prepare numerical and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ("location", OneHotEncoder(handle_unknown="ignore"), ["location"]),
        ("numeric", SimpleImputer(strategy="median"),
         ["total_sqft", "bath", "bhk"])
    ]
)

# Build the model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Data preparation completed!")

Training records: 10025
Testing records: 2507
Data preparation completed!


In [22]:
# Train the model
model.fit(X_train, y_train)

# Predict prices for test data
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Model Training Completed!")
print(f"Mean Absolute Error (MAE): {mae:.2f} lakhs")
print(f"R² Score: {r2:.4f}")
print(f"R² Percentage: {r2 * 100:.2f}%")

Model Training Completed!
Mean Absolute Error (MAE): 41.10 lakhs
R² Score: 0.3954
R² Percentage: 39.54%


In [23]:
# Start with the prepared dataset
improved_data = data.copy()

# Remove invalid or implausible property measurements
improved_data = improved_data[
    (improved_data["total_sqft"] >= 300) &
    (improved_data["total_sqft"] <= 10000) &
    (improved_data["price"] > 0) &
    (improved_data["bath"] > 0) &
    (improved_data["bhk"] > 0)
].copy()

# Calculate price per square foot (price is in lakhs)
improved_data["price_per_sqft"] = (
    improved_data["price"] * 100000 /
    improved_data["total_sqft"]
)

print("Original prepared records:", len(data))
print("Records after basic filtering:", len(improved_data))

print("\nPrice per square foot summary (₹):")
print(improved_data["price_per_sqft"].describe())

Original prepared records: 12532
Records after basic filtering: 12498

Price per square foot summary (₹):
count     12498.000000
mean       6773.268194
std        4624.423304
min         500.000000
25%        4302.082750
50%        5493.817245
75%        7407.407407
max      176470.588235
Name: price_per_sqft, dtype: float64


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Prepare features and target
X = improved_data[["location", "total_sqft", "bath", "bhk"]]
y = improved_data["price"]

# Use the same split for both models
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Preprocess location and numerical features
preprocessor = ColumnTransformer([
    ("location", OneHotEncoder(handle_unknown="ignore"), ["location"]),
    ("numeric", SimpleImputer(strategy="median"),
     ["total_sqft", "bath", "bhk"])
])

# Define both models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

results = {}

for name, regressor in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", regressor)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    results[name] = {
        "model": pipeline,
        "MAE": mae,
        "R2": r2
    }

    print(f"\n{name}")
    print(f"MAE: ₹{mae:.2f} lakhs")
    print(f"R² Score: {r2:.4f}")


Linear Regression
MAE: ₹44.66 lakhs
R² Score: 0.5188

Random Forest
MAE: ₹36.75 lakhs
R² Score: 0.5044


In [25]:
best_model = results["Random Forest"]["model"]

comparison = pd.DataFrame({
    "Actual Price (Lakhs)": y_test,
    "Predicted Price (Lakhs)": best_model.predict(X_test)
})

comparison["Error (Lakhs)"] = (
    comparison["Actual Price (Lakhs)"] -
    comparison["Predicted Price (Lakhs)"]
).abs()

comparison.head(10)

,Actual Price (Lakhs),Predicted Price (Lakhs),Error (Lakhs)
1811,110.0,100.323806,9.676194
10371,25.4,33.085092,7.685092
4951,155.0,145.886357,9.113643
6198,100.0,186.405742,86.405742
8330,77.0,59.735012,17.264988
2830,48.0,46.830141,1.169859
2938,150.0,114.384329,35.615671
5791,43.0,46.411788,3.411788
7461,60.0,50.030106,9.969894
9628,35.8,40.357272,4.557272


In [26]:
import joblib
from pathlib import Path

model_path = Path("../model/house_price_model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, model_path)

print("Model saved successfully!")
print("Location:", model_path.resolve())

Model saved successfully!
Location: C:\Users\KHAJA HUSSEIN\OneDrive\Desktop\Smart-Real-Estate-Price-Predictor\model\house_price_model.pkl
